RAG Pipeline - VectorDB to LLm Output Generation

In [4]:
import os ,sys
from dotenv import load_dotenv
load_dotenv()
print(os.getenv("GROQ_API-KEY"))
sys.path.insert(0, os.path.abspath(".."))


None


In [5]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage,SystemMessage

d:\Documents\PROJECTS\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class GroqLLM:
    def __init__(self,model_name:str = "llama-3.3-70b-versatile",api_key:str=None):
        """
        Initialize groq LLM
        Args:
            model_name:Groq model name (qwen2=-72b-instruct,llama3-70b-8192,etc.)
        """

        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError("Groq API key is required , Set GROQ_API_KEY enivironment variable or pass api_key_key parameter . ")
        
        self.llm=ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

        print(f"Initialized Groq LLM with model:{self.model_name}")
    
    def generate_response(self, query:str, context : str , max_length:int=500) -> str:
        """
        Generate response using retrived context

        Args:
            query:User question
            context: Retrieved document context
            max_length : maximum response length
        Returns:
            Generated response string 
        """

        #Create prompt template
        prompt_template=PromptTemplate(
            input_variables=["context","question"],
            template="""You are a helpful AI assistant .Use the following context to answer the question accurately and concisely.
            
            Context:{context}
            Question:{question}
            Answer: Provide a clear and informative answer based on the context above . If the context doesn't contain enough information to answer the question , say so."""
        )
        #Format the prompt 
        formatted_prompt = prompt_template.format(context=context, question=query)

        try :
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error generating response :{str(e)}"
    
    def generate_response_simple(self,query: str, context :str) -> str:
        """
        Simple response generation without complex prompting 
        
        Args:
            query : User question
            context:Retrieved context
        
        Respnse:
            Generated response 
        """
        simple_prompt = f"""Based on this context :{context} Question:{query} Answer: """

        try :
            messages=[HumanMessage(content=simple_prompt)]
            response=self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error :{str(e)}"
       

In [7]:
# Initialize Groq LLm (you'll need to set GROQ_API_KEY enviornment variable )
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully !!")
except ValueError as e:
    print(f"Warning:{e}")
    print("Please set your GROQ_API_KEY enviorment variable to use the LLM.")
    groq_llm = None
    

Initialized Groq LLM with model:llama-3.3-70b-versatile
Groq LLM initialized successfully !!


In [8]:
from rag_utils import RAGRetriever, VectorStore, embedding_manager

rag_retriever=RAGRetriever(VectorStore,embedding_manager)
rag_retriever.retrieve("Unified Multi-task Learning Framework")

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384
Vector store initialized.collection: pdf_documents
Existing documents in collection : 1182
Retrieving documents for query:'Unified Multi-task Learning Framework
Top K:5,Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.27it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_301e0455_225',
  'content': 'which provides additional information by consulting nearby\nobjects and surroundings (GBD-Net and multi-path).\n• Due to the existence of a large number of nonstandard\nsmall objects, the results on this dataset are much worse\nthan those of VOC 2007/2012. With the introduction of\nother powerful frameworks (e.g. ResNeXt [123]) and useful\nstrategies (e.g. multi-task learning [67], [124]), the perfor-\nmance can be improved.\n• The success of DSOD in training from scratch stresses the',
  'metadata': {'creationdate': '2019-04-17T00:45:22+00:00',
   'doc_index': 225,
   'title': '',
   'content_length': 481,
   'moddate': '2019-04-17T00:45:22+00:00',
   'creator': 'LaTeX with hyperref package',
   'format': 'PDF 1.5',
   'keywords': '',
   'producer': 'pdfTeX-1.40.17',
   'author': '',
   'subject': '',
   'file_path': '..\\data\\pdf\\1807.05511v2.pdf',
   'source': '..\\data\\pdf\\1807.05511v2.pdf',
   'creationDate': 'D:20190417004522Z',
   't

Integration Vectordb Context pipeline With LLM output 

In [9]:
##Simple Rag pipeline with Groq LLM 
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

groq_api_key= os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

def rag_simple(query,retriever,llm,top_k=3):
    ##retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content']for doc in results ])if results else ""

    if not context:
        return "No relevent context found to answer the question."
    
    ##generate the answer using Groq LLM
    prompt=f"""Use the following context to answer the questio concisely .
        Context:{context}
        question;{query}
        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query = query)])
    return response.content


In [10]:
answer=rag_simple("What is attention mechanism ? ",rag_retriever,llm)
print(answer)

Retrieving documents for query:'What is attention mechanism ? 
Top K:3,Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 46.34it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The attention mechanism is a process where individual attention heads learn to perform different tasks, exhibiting behavior related to the syntactic and semantic structure of sentences, and following long-distance dependencies in the encoder self-attention.


In [19]:
def rag_advanced(query, retriever,llm,top_k=5,min_score=0.2,return_context=False):
    """
    RAG pipeline with extra features :
    Return answer , sources,confidence score,and optionally full context.
    """
    results=retriever.retrieve(query,top_k=top_k,score_threshold=min_score)
    if not results:
        return {'answer':'No relavent context found','sources':[],'confidence':0.0,'context':' '}
    
    #prepare context and sources
    context="\n\n".join([doc['content']for doc in results])
    sources = [{
        'source':doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
        'page':doc['metadata'].get('page','unknown'),
        'score':doc['similarity_score'],
        'preview':doc['content'][:300]+ '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    #Generate answer 
    prompt = f"""Use the following context to answer the question concisely .\nContext :\n {context}\n \n Question:{query}\n \nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output={
        'answer':response.content,
        'sources':sources,
        'confidence':confidence
    }
    if return_context:
        output['context'] = context
    return output

result = rag_advanced("Deep learning in Salient Object Detection",rag_retriever,llm,top_k=3,min_score=0.1,return_context=True)
print("Answer:", result["answer"])
print("Sources:",result['sources'])
print("Confidence:",result['confidence'])
print("Context Preview:",result['context'][:300])


Retrieving documents for query:'Deep learning in Salient Object Detection
Top K:3,Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.44it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Due to its significance in providing high-level and multi-scale features, deep learning is used in salient object detection.
Sources: [{'source': '..\\data\\pdf\\1807.05511v2.pdf', 'page': 19, 'score': 0.6361712217330933, 'preview': 'THIS PAPER HAS BEEN ACCEPTED BY IEEE TRANSACTIONS ON NEURAL NETWORKS AND LEARNING SYSTEMS FOR PUBLICATION\n20\n[155] Q. Hou, M.-M. Cheng, X.-W. Hu, A. Borji, Z. Tu, and P. Torr,\n“Deeply supervised salient object detection with short connections,”\narXiv:1611.04849, 2016.\n[156] Q. Yan, L. Xu, J. Shi, an...'}, {'source': '..\\data\\pdf\\1807.05511v2.pdf', 'page': 19, 'score': 0.6361712217330933, 'preview': 'THIS PAPER HAS BEEN ACCEPTED BY IEEE TRANSACTIONS ON NEURAL NETWORKS AND LEARNING SYSTEMS FOR PUBLICATION\n20\n[155] Q. Hou, M.-M. Cheng, X.-W. Hu, A. Borji, Z. Tu, and P. Torr,\n“Deeply supervised salient object detection with short connections,”\narXiv:1611.04849, 2016.\n[156] Q. Yan, L. Xu, J. Shi, an...'}, {'source': '..\\data\\pdf\\1807.055

In [20]:
##Added Streaming ,citations ,History ,Summarization
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query:'what is attention is all you need
Top K:3,Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.67it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)



Final Answer: No relevant context found.
Summary: There is no text to summarize as the provided answer states "No relevant context found." This indicates that there is no information available to create a summary.
History: {'question': 'what is attention is all you need', 'answer': 'No relevant context found.', 'sources': [], 'summary': 'There is no text to summarize as the provided answer states "No relevant context found." This indicates that there is no information available to create a summary.'}
